# Recall Analysis Statistics

This Notebook is analysing the Data properly from our experiments: After Loading the Data and Computing all the features, here, we are making 

### Load Data


In [202]:
### Imports
import matplotlib.pyplot as plt

ANNAME = "TEST_withoutCenter_res100_dgv5_bll500_blb75_smw10_smL2000_swG50_trOall_trB0" ### WORK IN PROGRESS - add code to read gaze data
EXPNAME = "SecondExp"
### Parameters and Globals
CLASSN = 4                       # Number of MW Discrete Classess
DATAFOLDER = "data_second"       # Data Folder for experiment
LABELING = False                 # Labeling Mode
CLASSESS =  {1:"Focus",          # Classess Dictionairy t understand what means what
             2:"Task-Related Thoughts", 
             3: "Mind Wandering",
             4: "Mind Wandering"}

CONTINUE = True
VERBOSE = False
# FIlteri Data Parameters
SUBJECTS = "all"    
SHORTLEMMAS = True   # Whether to include Utterances that have simple to little Lemmas


VIZFOLDER = f"extrF_{CLASSN}_{SHORTLEMMAS}_{SUBJECTS}"


In [204]:
####    Load Necessery Variables and Paths:
import os 
from storyTools import loadStory,splitStoryEntity,reLabel
import pandas as pd
import numpy as np 
import pickle

### General Working, Data, and Saving Path:
workingDir = os.getcwd();                                 # Current Folder
path = os.path.split(workingDir)[0] + "\\" + DATAFOLDER   # Data path
savePathRecall = os.path.join(workingDir,EXPNAME,'Recalls')       # Save Path for Recall results
savePathFeatures = os.path.join(workingDir,EXPNAME,'RecallAnalysis',VIZFOLDER)

if os.path.isdir(savePathFeatures) == 0:
    os.makedirs(savePathFeatures) # Create a directory for recall results

subjects = [f for f in os.listdir(path) if  not os.path.isfile(os.path.join(path, f))]  # get all the filenames


### Get Polish Stopwords:
with open(r"C:\Users\barak\Documents\Python_Scripts\CoproraTools\polish.stopwords.txt", encoding="utf-8") as f:
    polish_stopwords = set(line.strip().lower() for line in f if line.strip())


### Load Previous Excel File with Data:
dfData = pd.read_excel(os.path.join(savePathFeatures,"data.xlsx"), index_col=0)
dfAnalysis = pd.read_excel(os.path.join(savePathFeatures,"features.xlsx"), index_col=0)

### Load Eye and BEhavioral Data:
dfEyeData = pd.read_excel(os.path.join(os.path.join(workingDir,EXPNAME,ANNAME,'summary'),"all_data.xlsx"),header=[0, 1, 2])

# Replace 'Unnamed' with empty string in all levels
dfEyeData.columns = pd.MultiIndex.from_tuples(
    tuple('' if "Unnamed" in str(x) else x for x in col)
    for col in dfEyeData.columns
)
dfEyeData.drop(0,axis=0,inplace=True)

In [205]:
### RenameSubjects:
subjPaths = [ f.name for f in os.scandir(savePathRecall) if f.is_dir() ]
import re
from itertools import zip_longest
dfData['DS'] = dfData['Subject']
dfAnalysis['DS'] = dfData['Subject']

fragmentEquivalenceDF = pd.read_excel( os.path.join(workingDir,EXPNAME,'RecallAnalysis',"fragEquiv.xlsx"))
for sn,s in enumerate(subjPaths):
    currSlice = dfData['Subject'] == s

    dfData['DS'][currSlice] = f'DS{int(sn+1):02d}'
    dfAnalysis['DS'][currSlice] = f'DS{int(sn+1):02d}'


dfData = dfData.merge(fragmentEquivalenceDF,on="Fragment")
dfAnalysis = dfAnalysis.merge(fragmentEquivalenceDF,on="Fragment")

In [206]:
def pad_columns(df, depth, fill=''):
    if not isinstance(df.columns, pd.MultiIndex):
        df.columns = pd.MultiIndex.from_tuples([(c,) + (fill,) * (depth - 1) for c in df.columns])
    elif df.columns.nlevels < depth:
        df.columns = pd.MultiIndex.from_tuples(
            [tuple(col) + (fill,) * (depth - len(col)) for col in df.columns]
            for col in df.columns
        )
    return df

# Pad dfAnalysis columns to 3 levels
dfAnalysis = pad_columns(dfAnalysis, 3)

# Now merge using the correct tuple for 'trialNum'
merged = dfEyeData.merge(dfAnalysis, on=[('DS', '', ''),('trialNum', '', '')])
merged

MW_Estimate GazeDifference Pupil_Diameter                           \
                                                Mean                      Std   
                                            Left_Eye    Right_Eye    Left_Eye   
0      0.0        37.0      63.806293    4586.951565  4396.278392  440.848169   
1      1.0         1.0      46.395256    5006.267608  4579.383238  226.684155   
2      2.0         2.0      45.062071    4912.174249  4429.950718  213.217401   
3      3.0         0.0      49.103675    4881.164341  4462.567599  320.822268   
4      4.0        27.0      54.162255    5126.331326  4534.991356  208.714549   
..     ...         ...            ...            ...          ...         ...   
207  275.0        94.0      28.812155    4700.086577  5259.923031  204.311675   
208  276.0         0.0      25.311283    4771.325979  5357.633029  205.847450   
209  277.0        22.0      22.763584    4637.627970  5179.476132  184.245992   
210  278.0         0.0      23.190695    4696.247165  5264.533808  166.981565   
211  279.0        10.0      33.601182    4823.519881  5457.284255  239.316007   

                                      Tracking  ...      Blinks            \
                     diff                       ...                         
      Right_Eye  Left_Eye Right_Eye             ... BlinkNumber BlinkRate   
0    348.593768 -0.111329 -0.074762  UNTRACKED  ...         3.0    0.0003   
1    197.586335  0.171510 -0.004754    TRACKED  ...         4.0    0.0004   
2    161.052121 -0.022151 -0.026559  UNTRACKED  ...         8.0    0.0008   
3    258.690640 -0.181888 -0.178782    TRACKED  ...         4.0    0.0004   
4    179.720664 -0.062850 -0.069350  UNTRACKED  ...         7.0    0.0007   
..          ...       ...       ...        ...  ...         ...       ...   
207  269.894204 -0.083365 -0.159815  UNTRACKED  ...         7.0    0.0007   
208  283.115156 -0.283584 -0.372190    TRACKED  ...         3.0    0.0003   
209  262.842562 -0.192223 -0.255434  UNTRACKED  ...         3.0    0.0003   
210  228.164516 -0.141152 -0.228232    TRACKED  ...         5.0    0.0005   
211  328.542572 -0.098263 -0.109166  UNTRACKED  ...        10.0    0.0010   

                                    Subject            Fragment Attention  \
                                                                            
    BlinkDuration BlinkDurationSd                                           
0      159.333333       70.698106  ADNA2507   recall_KAROLINA_1         3   
1      115.500000       15.960890  ADNA2507      recall_JANEK_1         1   
2      147.000000       22.045408  ADNA2507   recall_KAROLINA_2         1   
3      129.000000       26.888659  ADNA2507      recall_JANEK_2         1   
4      133.142857       13.973736  ADNA2507   recall_KAROLINA_3         1   
..            ...             ...       ...                 ...       ...   
207    140.285714       32.836018  PIKR2907     recall_JANEK_18         3   
208    164.666667      150.149555  PIKR2907  recall_KAROLINA_19         1   
209    138.666667       27.535835  PIKR2907     recall_JANEK_19         3   
210    145.600000       16.989408  PIKR2907  recall_KAROLINA_20         1   
211    153.400000       18.089776  PIKR2907     recall_JANEK_20         3   

    utt_leng Unnamed: 0.1 Unnamed: 0  
                                      
                                      
0        118            0         20  
1        121            1          0  
2        216            2         31  
3        116            3         11  
4        154            4         33  
..       ...          ...        ...  
207      328           35          9  
208       65           36         30  
209      168           37         10  
210       64           38         32  
211      170           39         12  

[212 rows x 40 columns]

In [207]:
keys = ["Subject", "trialNum"]

# de-dup keys (important if merged is m:1)
ana_keys = dfAnalysis[keys].drop_duplicates()
mer_keys = merged[keys].drop_duplicates()

# 1) In dfAnalysis but NOT in merged
missing_keys = (
    pd.MultiIndex.from_frame(ana_keys)
    .difference(pd.MultiIndex.from_frame(mer_keys))
)
missing_from_merged = (
    dfAnalysis.set_index(keys).loc[missing_keys].reset_index()
)

missing_from_merged

,Subject,trialNum,Fragment,Attention,utt_leng,DS,Unnamed: 0.1,Unnamed: 0
,,,,,,,,
,,,,,,,,
